# LLM evaluation & benchmarks

Evaluation turns fuzzy model behavior into measurements with uncertainty and failure modes.

Benchmarks need task outcomes, uncertainty, pass@k conventions, contamination audits, and cost terms before they can support a release decision. Save a copy to Drive to edit.

In [ ]:

import math
import random
from collections import Counter

import matplotlib.pyplot as plt
import numpy as np

random.seed(9)
np.random.seed(9)

RUNG_NAMES = ["D1", "D2", "D3", "D4", "D5"]


def softmax(logits):
    values = np.array(logits, dtype=float)
    shifted = values - values.max()
    weights = np.exp(shifted)
    return weights / weights.sum()


def tokenize(text):
    clean = "".join(ch.lower() if ch.isalnum() else " " for ch in text)
    return [tok for tok in clean.split() if tok]


def cosine(a, b):
    left = np.array(a, dtype=float)
    right = np.array(b, dtype=float)
    denom = np.linalg.norm(left) * np.linalg.norm(right)
    if denom == 0:
        return 0.0
    return float(np.dot(left, right) / denom)


def bow_vector(text, vocab):
    counts = Counter(tokenize(text))
    return np.array([counts.get(term, 0) for term in vocab], dtype=float)


def preview_ladder(ladder):
    for rung in ladder:
        name = rung["name"]
        size = len(rung.get("tasks", rung.get("items", rung.get("queries", []))))
        note = rung.get("note", "")
        print(f"{name}: size={size} {note}")
        sample_key = "tasks" if "tasks" in rung else "items" if "items" in rung else "queries"
        print(rung[sample_key][0])


## The concept, built once

The lesson formula is $\hat p\pm1.96\sqrt{\hat p(1-\hat p)/n},\quad \mathrm{pass@}k=1-\frac{\binom{n-c}{k}}{\binom{n}{k}}$. We compute the benchmark statistic itself rather than reading a leaderboard. Accuracy is paired with a confidence interval so random noise is visible.

In [ ]:

def binomial_interval(successes, n):
    p_hat = successes / n
    half_width = 1.96 * math.sqrt(p_hat * (1 - p_hat) / n)
    return p_hat, half_width


def pass_at_k(n, c, k):
    if n - c < k:
        return 1.0
    return 1 - math.comb(n - c, k) / math.comb(n, k)


The reusable evaluator adds pass@k, pairwise win rate, contamination, and cost weighting. The asserts lock the lesson's exact numbers, including $72/100=0.720$ and pass@3 $0.708$.

In [ ]:

def evaluate_benchmark(successes, total, n_candidates, correct_candidates, k, wins, losses, contaminated, cost, base_score):
    p_hat, half_width = binomial_interval(successes, total)
    pass_k = pass_at_k(n_candidates, correct_candidates, k)
    win_rate = wins / (wins + losses)
    contamination_rate = contaminated / total
    cost_weighted = base_score - 0.01 * cost
    return {"accuracy": p_hat, "half_width": half_width, "pass_k": pass_k, "win_rate": win_rate, "contamination": contamination_rate, "cost_weighted": cost_weighted}


lesson_eval = evaluate_benchmark(72, 100, 10, 3, 3, 30, 20, 5, 20, 0.8)

assert round(lesson_eval["accuracy"], 3) == 0.720
assert round(lesson_eval["half_width"], 3) == 0.088
assert round(lesson_eval["pass_k"], 3) == 0.708
assert round(lesson_eval["win_rate"], 3) == 0.600
assert round(lesson_eval["contamination"], 2) == 0.05
assert round(lesson_eval["cost_weighted"], 3) == 0.600
print(lesson_eval)


## The dataset ladder

Build D1-D5 inline so the notebook is self-contained in Colab. Each rung adds scale or a realistic failure mode while keeping CPU-only toy data.

In [ ]:

def make_eval_ladder():
    d1 = [{"successes": 1, "total": 1, "cost": 5, "contaminated": 0}]
    d2 = d1 + [{"successes": 7, "total": 10, "cost": 8, "contaminated": 0}]
    d3 = d2 + [{"successes": 13, "total": 20, "cost": 12, "contaminated": 1}]
    d4 = d3 + [{"successes": 30, "total": 40, "cost": 15, "contaminated": 1}]
    d5 = d4 + [{"successes": 72, "total": 100, "cost": 20, "contaminated": 5}]
    return [{"name": "D1", "items": d1, "note": "one benchmark item"}, {"name": "D2", "items": d2, "note": "clean rising task count"}, {"name": "D3", "items": d3, "note": "noisy benchmark"}, {"name": "D4", "items": d4, "note": "real-style QA/code eval"}, {"name": "D5", "items": d5, "note": "contaminated long benchmark"}]


eval_ladder = make_eval_ladder()
preview_ladder(eval_ladder)


## Run the same method across D1-D5

The metric follows the plan: task success, retrieval recall, unsupported rate, benchmark accuracy, or safety pass-rate depending on the topic.

In [ ]:

eval_rows = []
for rung in eval_ladder:
    latest = rung["items"][-1]
    result = evaluate_benchmark(latest["successes"], latest["total"], 10, 3, 3, 30, 20, latest["contaminated"], latest["cost"], 0.8)
    eval_rows.append({"rung": rung["name"], "metric": result["accuracy"], "half_width": result["half_width"], "contamination": result["contamination"]})

for row in eval_rows:
    print(row)


## Results visualization

The closing figure uses small multiples for the per-rung artifact and one summary curve across D1-D5.

In [ ]:

fig, axes = plt.subplots(2, 5, figsize=(16, 6))
for index, rung in enumerate(eval_ladder):
    item = rung["items"][-1]
    p_hat, half_width = binomial_interval(item["successes"], item["total"])
    axes[0, index].bar(["accuracy"], [p_hat], yerr=[half_width], color="slateblue")
    axes[0, index].set_ylim(0, 1.05)
    axes[0, index].set_title(rung["name"])

axes[1, 0].errorbar([row["rung"] for row in eval_rows], [row["metric"] for row in eval_rows], yerr=[row["half_width"] for row in eval_rows], marker="o")
axes[1, 0].set_ylim(0, 1.05)
axes[1, 0].set_ylabel("accuracy with CI")
axes[1, 0].set_title("uncertainty vs sample size")
for blank in axes[1, 1:]:
    blank.axis("off")
plt.tight_layout()


## Pitfall on the hardest rung

Reproduce the named D5 failure, then turn on the fix and compare the behavior.

In [ ]:

model_a = binomial_interval(72, 100)
model_b = binomial_interval(75, 100)
contamination_flag = eval_ladder[-1]["items"][-1]["contaminated"] > 0
print("A interval", model_a)
print("B interval", model_b)
print("intervals overlap", model_a[0] + model_a[1] >= model_b[0] - model_b[1])
print("contamination flag", contamination_flag)


## Evaluate it + Practice

- Compare the planned metric with a no-skill baseline such as answering without tools, retrieval, claims, intervals, or guardrails.
- Sanity-check one hand-computable lesson number before trusting the ladder.
- Ablate the key idea and verify the metric drops or the failure signal rises.
- Watch failure signals: retry loops, irrelevant evidence, hidden unsupported claims, noisy rank changes, or unsafe tool actions.

Practice prompts:
- Compare 72/100 with 75/100 and decide whether the confidence intervals justify a release.
- Change pass@3 to pass@1 and pass@5 while keeping n=10 and c=3.
- Add a dollar or latency cost term and find when the cheaper model wins.
